In [ ]:
import torch
from torch import nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt


class VGG16(nn.Module):
  def __init__(self, input_shape: int, output_shape: int):
    super().__init__()
    self.block1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape, out_channels=64, kernel_size=3, padding=1, stride=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1, stride=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
    )

    self.block2 = nn.Sequential(
        nn.Conv2d(in_channels= 64, out_channels=128, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(in_channels= 128, out_channels=128, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding =0)
    )
    self.block3 = nn.Sequential(
        nn.Conv2d(in_channels= 128, out_channels=256, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(in_channels= 256, out_channels=256, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(in_channels= 256, out_channels=256, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding =0)
    )

    self.block4 = nn.Sequential(
        nn.Conv2d(in_channels = 256, out_channels= 512, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.Conv2d(in_channels = 512, out_channels= 512, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.Conv2d(in_channels = 512, out_channels= 512, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
    )

    self.block5 = nn.Sequential(
        nn.Conv2d(in_channels = 512, out_channels= 512, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.Conv2d(in_channels = 512, out_channels= 512, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.Conv2d(in_channels = 512, out_channels= 512, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features= 512*1*1, out_features=4096),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(in_features=4096, out_features=4096),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(in_features=4096, out_features=output_shape)
    )

  def forward(self, x):
    x = self.block1(x)
    x = self.block2(x)
    x = self.block3(x)
    x = self.block4(x)
    x = self.block5(x)

    x= self.classifier(x)

    return x



In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

In [ ]:
train_data = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
test_data = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=2)


In [ ]:
model = VGG16(input_shape=3, output_shape=10)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model.parameters(),lr=0.01, momentum=0.9, weight_decay=5e-4)

# Check for GPU and move model to device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Using device: {device}")

Using device: cuda


In [ ]:
def training_loop(model, dataloader, loss_fn, optimizer):
  model.train()
  train_loss, train_acc =0,0
  for x, y in dataloader:
    # Move data to device
    x, y = x.to(device), y.to(device)

    y_pred = model(x)
    loss = loss_fn(y_pred, y)

    train_loss += loss.item() # Use .item() for scalar values
    train_acc += (y_pred.argmax(dim=1) == y).sum().item() / len(y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  train_loss /= len(dataloader)
  train_acc /= len(dataloader)
  return train_loss, train_acc


def test_loop(model, dataloader, loss_fn):
  model.eval()
  test_loss, test_acc =0,0
  with torch.inference_mode():
    for x, y in dataloader:
      x, y = x.to(device), y.to(device)

      y_pred = model(x)
      loss = loss_fn(y_pred, y)
      test_loss += loss.item()
      test_acc += (y_pred.argmax(dim=1) == y).sum().item()/len(y)

  test_loss /= len(dataloader)
  test_acc /= len(dataloader)
  return test_loss, test_acc

In [ ]:

epochs = 20
for epoch in range(epochs):
  train_loss, train_acc = training_loop(model, train_loader, loss_fn, optimizer)
  test_loss, test_acc = test_loop(model, test_loader, loss_fn)
  print(f"Epoch {epoch+1}/{epochs} | "
          f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f} | "
          f"Test loss: {test_loss:.4f}, Test acc: {test_acc:.4f}")

Epoch 1/20 | Train loss: 1.5118, Train acc: 0.4390 | Test loss: 1.3814, Test acc: 0.5202
Epoch 2/20 | Train loss: 0.9746, Train acc: 0.6576 | Test loss: 1.0147, Test acc: 0.6630
Epoch 3/20 | Train loss: 0.7638, Train acc: 0.7393 | Test loss: 0.7715, Test acc: 0.7448
Epoch 4/20 | Train loss: 0.6566, Train acc: 0.7783 | Test loss: 0.6273, Test acc: 0.7928
Epoch 5/20 | Train loss: 0.5702, Train acc: 0.8077 | Test loss: 0.5792, Test acc: 0.8087
Epoch 6/20 | Train loss: 0.5201, Train acc: 0.8257 | Test loss: 0.6189, Test acc: 0.7891
Epoch 7/20 | Train loss: 0.4766, Train acc: 0.8414 | Test loss: 0.5448, Test acc: 0.8176
Epoch 8/20 | Train loss: 0.4338, Train acc: 0.8550 | Test loss: 0.4563, Test acc: 0.8450
Epoch 9/20 | Train loss: 0.4013, Train acc: 0.8648 | Test loss: 0.4669, Test acc: 0.8418
Epoch 10/20 | Train loss: 0.3802, Train acc: 0.8713 | Test loss: 0.4286, Test acc: 0.8582
Epoch 11/20 | Train loss: 0.3526, Train acc: 0.8793 | Test loss: 0.4509, Test acc: 0.8523
Epoch 12/20 | Train